## 1. NIFTY 50 Constituent Universe

Source: NSE India
Snapshot date: 2026-10-28

The constituent list is frozen for this project.
The use of a current constituent list for historical
analysis introduces survivorship bias.

In [2]:
from pathlib import Path
from io import BytesIO
import json
import requests
import pandas as pd
from datetime import datetime
import yfinance as yf

In [1]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "universe"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Official NSE constituent CSV
URL = "https://nsearchives.nseindia.com/content/indices/ind_nifty50list.csv"

response = requests.get(
    URL,
    headers={
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/csv,*/*",
    },
    timeout=30,
)

response.raise_for_status()

# Preserve the original downloaded file unchanged
raw_path = RAW_DIR / "nifty50_constituents_2026-10-28.csv"
raw_path.write_bytes(response.content)

# Read a separate in-memory copy for validation
constituents = pd.read_csv(BytesIO(response.content))

# Basic validation
assert "Symbol" in constituents.columns
assert len(constituents) == 50
assert constituents["Symbol"].notna().all()
assert constituents["Symbol"].is_unique

# Create Yahoo Finance tickers separately
constituents["Yahoo_Ticker"] = (
    constituents["Symbol"].astype(str) + ".NS"
)

# Record source metadata
metadata = {
    "source": "NSE India",
    "source_url": URL,
    "snapshot_date": "2026-10-28",
    "retrieved_at": datetime.now().astimezone().isoformat(),
    "constituent_count": len(constituents),
    "raw_file": raw_path.name,
    "survivorship_bias": True,
}

metadata_path = RAW_DIR / "nifty50_universe_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Constituent count:", len(constituents))
print("Unique symbols:", constituents["Symbol"].nunique())
print("Raw file:", raw_path)
print("Metadata:", metadata_path)

display(
    constituents[
        ["Company Name", "Symbol", "Yahoo_Ticker"]
    ].head(10)
)

Constituent count: 50
Unique symbols: 50
Raw file: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/universe/nifty50_constituents_2026-10-28.csv
Metadata: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/universe/nifty50_universe_metadata.json


,Company Name,Symbol,Yahoo_Ticker
0,Adani Enterprises Ltd.,ADANIENT,ADANIENT.NS
1,Adani Ports and Special Economic Zone Ltd.,ADANIPORTS,ADANIPORTS.NS
2,Apollo Hospitals Enterprise Ltd.,APOLLOHOSP,APOLLOHOSP.NS
3,Asian Paints Ltd.,ASIANPAINT,ASIANPAINT.NS
4,Axis Bank Ltd.,AXISBANK,AXISBANK.NS
5,BSE Ltd.,BSE,BSE.NS
6,Bajaj Auto Ltd.,BAJAJ-AUTO,BAJAJ-AUTO.NS
7,Bajaj Finance Ltd.,BAJFINANCE,BAJFINANCE.NS
8,Bajaj Finserv Ltd.,BAJAJFINSV,BAJAJFINSV.NS
9,Bharat Electronics Ltd.,BEL,BEL.NS


In [ ]:
current_dir = Path.cwd()
PROJECT_ROOT = next(
    (
        path for path in [current_dir, *current_dir.parents]
        if (path / "PROTOCOL.md").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

UNIVERSE_DIR = PROJECT_ROOT / "data" / "raw" / "universe"

STOCK_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "stocks"
MACRO_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "market_macro"

STOCK_RAW_DIR.mkdir(parents=True, exist_ok=True)
MACRO_RAW_DIR.mkdir(parents=True, exist_ok=True)


# 2. LOAD FROZEN STOCK UNIVERSE
universe_file = (
    UNIVERSE_DIR / "nifty50_constituents_2026-10-28.csv"
)

universe = pd.read_csv(universe_file)

stock_tickers = (
    universe["Symbol"].astype(str) + ".NS"
).tolist()

macro_tickers = {
    "^NSEI": "NIFTY_50",
    "^INDIAVIX": "INDIA_VIX",
    "INR=X": "USD_INR",
    "BZ=F": "BRENT_CRUDE",
    "^TNX": "US_10Y_YIELD"
}

print("Stocks:", len(stock_tickers))
print("Additional market series:", len(macro_tickers))


# 3. DOWNLOAD FUNCTION
START_DATE = "2010-01-01"

download_log = []
def download_batch(tickers, output_dir, asset_type):

    data = yf.download(
        tickers=tickers,
        start=START_DATE,
        interval="1d",
        auto_adjust=False,
        actions=True,
        group_by="ticker",
        threads=True,
        progress=False,
        multi_level_index=True
    )

    for ticker in tickers:

        try:
            ticker_data = data[ticker].copy()

            if ticker_data.empty or ticker_data.isna().all().all():
                raise ValueError("No usable data returned.")

            # Preserve the downloaded observations.
            safe_name = (
                ticker.replace("^", "")
                .replace("=", "_")
                .replace(".", "_")
            )

            output_path = output_dir / f"{safe_name}.parquet"

            ticker_data.to_parquet(output_path)

            download_log.append({
                "ticker": ticker,
                "asset_type": asset_type,
                "status": "success",
                "rows": len(ticker_data),
                "first_date": ticker_data.index.min(),
                "last_date": ticker_data.index.max(),
                "file": str(output_path)
            })

        except Exception as error:

            download_log.append({
                "ticker": ticker,
                "asset_type": asset_type,
                "status": "failed",
                "rows": 0,
                "first_date": None,
                "last_date": None,
                "file": None,
                "error": str(error)
            })

# 4. DOWNLOAD STOCK DATA IN BATCHES
BATCH_SIZE = 10

for start in range(0, len(stock_tickers), BATCH_SIZE):
    batch = stock_tickers[start:start + BATCH_SIZE]
    print(f"Downloading stocks {start + 1}–{start + len(batch)}")
    download_batch(
        tickers=batch,
        output_dir=STOCK_RAW_DIR,
        asset_type="stock"
    )


# 5. DOWNLOAD MARKET AND MACRO DATA
download_batch(
    tickers=list(macro_tickers.keys()),
    output_dir=MACRO_RAW_DIR,
    asset_type="market_macro"
)

# saving the manifest
manifest = pd.DataFrame(download_log)
manifest_path = PROJECT_ROOT / "data" / "raw" / "download_manifest.csv"
manifest.to_csv(manifest_path, index=False)


print("\nDOWNLOAD SUMMARY")

print(manifest.groupby(
    ["asset_type", "status"]
).size())

print("\nFailed downloads:")

display(
    manifest[manifest["status"] == "failed"]
)

print("\nManifest saved:", manifest_path)


Stocks: 50
Additional market series: 5


/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword

/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword


DOWNLOAD SUMMARY
asset_type    status 
market_macro  success     5
stock         failed      1
              success    49
dtype: int64

Failed downloads:


/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/.venv/lib/python3.13/site-packages/yfinance/multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)


,ticker,asset_type,status,rows,first_date,last_date,file,error
3,ASIANPAINT.NS,stock,failed,0,NaT,NaT,NaN,No usable data returned.



Manifest saved: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/download_manifest.csv


In [5]:
ticker = "ASIANPAINT.NS"

safe_name = ticker.replace("^", "").replace("=", "_").replace(".", "_")
output_path = STOCK_RAW_DIR / f"{safe_name}.parquet"

# Confirm the downloaded file exists
if not output_path.exists():
    raise FileNotFoundError(
        f"Downloaded file not found: {output_path}"
    )

# Read the saved data
retry_data = pd.read_parquet(output_path)

if retry_data.empty:
    raise ValueError("The saved Parquet file is empty.")

# Update manifest
manifest_path = PROJECT_ROOT / "data" / "raw" / "download_manifest.csv"
manifest = pd.read_csv(manifest_path)

mask = manifest["ticker"] == ticker

manifest.loc[mask, "status"] = "success"
manifest.loc[mask, "rows"] = len(retry_data)
manifest.loc[mask, "first_date"] = str(retry_data.index.min().date())
manifest.loc[mask, "last_date"] = str(retry_data.index.max().date())
manifest.loc[mask, "file"] = str(output_path)
manifest.loc[mask, "error"] = None

manifest.to_csv(manifest_path, index=False)

print("Manifest updated.")
print("Ticker:", ticker)
print("Rows:", len(retry_data))
print("First date:", retry_data.index.min().date())
print("Last date:", retry_data.index.max().date())

display(manifest.groupby(["asset_type", "status"]).size())

Manifest updated.
Ticker: ASIANPAINT.NS
Rows: 4159
First date: 2010-01-04
Last date: 2026-11-06


asset_type    status 
market_macro  success     5
stock         success    50
dtype: int64

In [11]:
manifest_path = PROJECT_ROOT / "data" / "raw" / "download_manifest.csv"
manifest = pd.read_csv(manifest_path)

audit_results = []

# Audit each dataset 
for _, item in manifest.iterrows():

    file_path = Path(item["file"]) if pd.notna(item["file"]) else None

    result = {
        "ticker": item["ticker"],
        "asset_type": item["asset_type"],
        "file_exists": False,
        "rows": 0,
        "first_date": None,
        "last_date": None,
        "duplicate_dates": None,
        "missing_values": None,
        "invalid_prices": None,
        "invalid_ohlc": None,
        "negative_volume": None,
        "audit_status": "failed"
    }

    try:
        if file_path is None or not file_path.exists():
            raise FileNotFoundError("Parquet file not found.")

        data = pd.read_parquet(file_path)

        if data.empty:
            raise ValueError("Dataset is empty.")

        result["file_exists"] = True
        result["rows"] = len(data)

        # Date coverage
        dates = pd.to_datetime(data.index, errors="coerce")

        result["first_date"] = str(dates.min().date())
        result["last_date"] = str(dates.max().date())

        # Duplicate dates
        result["duplicate_dates"] = int(data.index.duplicated().sum())

        # Missing values by column
        missing = data.isna().sum()
        result["missing_values"] = json.dumps(
            missing[missing > 0].to_dict()
        )

        # Price validation
        price_columns = ["Open", "High", "Low", "Close"]
        available_prices = [
            col for col in price_columns if col in data.columns
        ]

        if "Close" not in data.columns:
            raise ValueError("Close price column is missing.")

        prices = data[available_prices]

        result["invalid_prices"] = int(
            (prices <= 0).any(axis=1).sum()
        )

        # OHLC consistency: High must be >= all other prices,
        # and Low must be <= all other prices.
        if all(col in data.columns for col in price_columns):
            ohlc = data[price_columns].apply(
                pd.to_numeric,
                errors="coerce"
            )

            invalid_ohlc = (
                (data["High"] < data["Low"])
                | (data["High"] < data["Open"])
                | (data["High"] < data["Close"])
                | (data["Low"] > data["Open"])
                | (data["Low"] > data["Close"])
            )

            result["invalid_ohlc"] = int(invalid_ohlc.sum())

        # Volume validation
        if "Volume" in data.columns:
            result["negative_volume"] = int(
                (data["Volume"] < 0).sum()
            )

        result["audit_status"] = "success"

    except Exception as error:
        result["error"] = str(error)

    audit_results.append(result)


# Audit Report
audit = pd.DataFrame(audit_results)

audit_path = PROJECT_ROOT / "data" / "raw" / "data_quality_audit.csv"
audit.to_csv(audit_path, index=False)


# Results
print("AUDIT SUMMARY")
print(audit["audit_status"].value_counts())

print("\nDATASET COVERAGE")
display(
    audit[
        [
            "ticker",
            "asset_type",
            "rows",
            "first_date",
            "last_date"
        ]
    ]
)

print("\nDATA QUALITY ISSUES")

issue_columns = [
    "duplicate_dates",
    "invalid_prices",
    "invalid_ohlc",
    "negative_volume"
]

display(
    audit[
        audit[issue_columns].fillna(0).gt(0).any(axis=1)
        | audit["missing_values"].fillna("{}").ne("{}")
        | audit["audit_status"].ne("success")
    ]
)

print("\nFull audit saved:", audit_path)

AUDIT SUMMARY
audit_status
success    55
Name: count, dtype: int64

DATASET COVERAGE


,ticker,asset_type,rows,first_date,last_date
0,ADANIENT.NS,stock,4159,2010-01-04,2026-11-06
1,ADANIPORTS.NS,stock,4159,2010-01-04,2026-11-06
2,APOLLOHOSP.NS,stock,4159,2010-01-04,2026-11-06
3,ASIANPAINT.NS,stock,4159,2010-01-04,2026-11-06
4,AXISBANK.NS,stock,4159,2010-01-04,2026-11-06
5,BSE.NS,stock,4159,2010-01-04,2026-11-06
6,BAJAJ-AUTO.NS,stock,4159,2010-01-04,2026-11-06
7,BAJFINANCE.NS,stock,4159,2010-01-04,2026-11-06
8,BAJAJFINSV.NS,stock,4159,2010-01-04,2026-11-06
9,BEL.NS,stock,4159,2010-01-04,2026-11-06



DATA QUALITY ISSUES


,ticker,asset_type,file_exists,rows,first_date,last_date,duplicate_dates,missing_values,invalid_prices,invalid_ohlc,negative_volume,audit_status
5,BSE.NS,stock,True,4159,2010-01-04,2026-11-06,0,"{""Open"": 1748, ""High"": 1748, ""Low"": 1748, ""Clo...",0,0,0,success
10,BHARTIARTL.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
11,CIPLA.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
12,COALINDIA.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 212, ""High"": 212, ""Low"": 212, ""Close""...",0,0,0,success
13,DRREDDY.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
14,EICHERMOT.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
15,ETERNAL.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 2851, ""High"": 2851, ""Low"": 2851, ""Clo...",0,0,0,success
16,GRASIM.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
18,HDFCBANK.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1, ""High"": 1, ""Low"": 1, ""Close"": 1, ""...",0,0,0,success
19,HDFCLIFE.NS,stock,True,4160,2010-01-04,2026-11-06,0,"{""Open"": 1944, ""High"": 1944, ""Low"": 1944, ""Clo...",0,0,0,success



Full audit saved: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/data_quality_audit.csv


In [14]:
for ticker in ["INR=X", "BZ=F"]:

    safe_name = ticker.replace("^", "").replace("=", "_").replace(".", "_")
    file_path = MACRO_RAW_DIR / f"{safe_name}.parquet"

    data = pd.read_parquet(file_path)

    # Remove the Price column level if present
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(-1)

    ohlc = data[["Open", "High", "Low", "Close"]]

    invalid = (
        (ohlc["High"] < ohlc["Low"])
        | (ohlc["High"] < ohlc["Open"])
        | (ohlc["High"] < ohlc["Close"])
        | (ohlc["Low"] > ohlc["Open"])
        | (ohlc["Low"] > ohlc["Close"])
    )

    print(f"\n{ticker}: {invalid.sum()} invalid observations")

    display(
        data.loc[
            invalid,
            ["Open", "High", "Low", "Close", "Volume"]
        ].head(15)
    )


INR=X: 232 invalid observations


Price,Open,High,Low,Close,Volume
Date,,,,,
2010-08-19,46.257999,46.498001,46.257999,46.537998,0.0
2010-09-30,44.563000,44.563000,44.563000,44.689999,0.0
2010-11-05,43.888000,43.900002,43.840000,44.013000,0.0
2010-11-08,44.205002,44.452999,44.205002,43.900002,0.0
2010-11-17,45.077999,45.167999,45.013000,45.189999,0.0
2010-11-23,45.410000,45.728001,45.410000,45.299999,0.0
2010-11-24,45.549999,45.873001,45.549999,45.540001,0.0
2010-12-24,45.160000,45.240002,45.073002,45.060001,0.0
2010-12-27,45.110001,45.278000,45.070000,45.009998,0.0



BZ=F: 30 invalid observations


Price,Open,High,Low,Close,Volume
Date,,,,,
2010-01-04,80.120003,79.820000,80.120003,80.120003,97.0
2010-01-05,80.589996,80.260002,80.589996,80.589996,97.0
2010-01-06,81.129997,81.699997,80.129997,81.889999,7.0
2010-01-07,81.510002,81.510002,81.629997,81.510002,7.0
2010-01-11,80.970001,81.709999,81.010002,80.970001,7.0
2010-01-14,77.820000,78.470001,78.010002,77.820000,6.0
2010-04-14,85.180000,85.199997,84.860001,86.150002,15.0
2010-04-15,87.169998,87.129997,87.169998,87.169998,14.0
2010-04-29,87.250000,87.550003,86.919998,86.900002,22.0


In [16]:
# Missing Trading Date Assessment

nifty_file = MACRO_RAW_DIR / "NSEI.parquet"

nifty_data = pd.read_parquet(nifty_file)

nifty_dates = pd.DatetimeIndex(
    pd.to_datetime(nifty_data.index)
).normalize()

nifty_dates = nifty_dates.unique().sort_values()

print("NIFTY 50 trading dates:", len(nifty_dates))


missing_date_results = []

for _, item in manifest.iterrows():

    file_path = Path(item["file"])
    data = pd.read_parquet(file_path)

    # Normalize dates to remove time components
    available_dates = pd.DatetimeIndex(
        pd.to_datetime(data.index)
    ).normalize().unique()

    first_date = available_dates.min()
    last_date = available_dates.max()

    # Expected NIFTY trading dates within this dataset's coverage
    expected_dates = nifty_dates[
        (nifty_dates >= first_date)
        & (nifty_dates <= last_date)
    ]

    missing_dates = expected_dates.difference(available_dates)

    missing_date_results.append({
        "ticker": item["ticker"],
        "asset_type": item["asset_type"],
        "expected_trading_dates": len(expected_dates),
        "available_dates": len(available_dates),
        "missing_trading_dates": len(missing_dates),
        "missing_date_examples": [
            str(date.date()) for date in missing_dates[:10]
        ]
    })


# --------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------

missing_date_audit = pd.DataFrame(missing_date_results)

print("DATASETS WITH MISSING TRADING DATES")

display(
    missing_date_audit[
        missing_date_audit["missing_trading_dates"] > 0
    ].sort_values(
        "missing_trading_dates",
        ascending=False
    )
)

# Save the focused report
missing_date_path = (
    PROJECT_ROOT / "data" / "raw" / "missing_trading_dates.csv"
)

missing_date_audit.to_csv(missing_date_path, index=False)

print("\nReport saved:", missing_date_path)

NIFTY 50 trading dates: 4396
DATASETS WITH MISSING TRADING DATES


,ticker,asset_type,expected_trading_dates,available_dates,missing_trading_dates,missing_date_examples
0,ADANIENT.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
37,RELIANCE.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
27,JIOFIN.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
28,KOTAKBANK.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
29,LT.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
30,M&M.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
31,MARUTI.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
32,MAXHEALTH.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
33,NTPC.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."
34,NESTLEIND.NS,stock,4394,4159,237,"[2010-01-26, 2010-02-12, 2010-03-01, 2010-03-2..."



Report saved: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/data/raw/missing_trading_dates.csv


In [18]:
# Investigate Missing Date Pattern

sample_tickers = [
    "RELIANCE.NS",
    "JIOFIN.NS",
    "MAXHEALTH.NS"
]

# Inspect the NIFTY reference calendar
nifty_file = MACRO_RAW_DIR / "NSEI.parquet"
nifty_data = pd.read_parquet(nifty_file)

if isinstance(nifty_data.columns, pd.MultiIndex):
    nifty_data.columns = nifty_data.columns.get_level_values(-1)

print("NIFTY 50 reference calendar")
print("Rows:", len(nifty_data))
print("First date:", nifty_data.index.min())
print("Last date:", nifty_data.index.max())

# Inspect representative stock datasets
for ticker in sample_tickers:

    safe_name = ticker.replace("^", "").replace("=", "_").replace(".", "_")
    file_path = STOCK_RAW_DIR / f"{safe_name}.parquet"

    data = pd.read_parquet(file_path)

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(-1)

    valid_close = data["Close"].dropna()

    print(f"\n{'=' * 45}")
    print("Ticker:", ticker)
    print("Total rows:", len(data))
    print("Rows with valid Close:", len(valid_close))
    print("Rows with missing Close:", data["Close"].isna().sum())

    if not valid_close.empty:
        print("First valid Close:", valid_close.index.min())
        print("Last valid Close:", valid_close.index.max())

    # Show the first 10 rows with missing Close
    print("\nFirst missing Close observations:")
    display(data.loc[data["Close"].isna()].head(10))

# Inspect dates identified as missing in the previous audit
print("\nNIFTY reference data for the first reported missing dates:")

example_dates = missing_date_audit.loc[
        missing_date_audit["ticker"] == "RELIANCE.NS",
        "missing_date_examples"
    ].iloc[0]

display(
    nifty_data.loc[
        pd.to_datetime(nifty_data.index).normalize().isin(
            pd.to_datetime(example_dates).normalize()
        )
    ]
)

NIFTY 50 reference calendar
Rows: 4396
First date: 2010-01-01 00:00:00
Last date: 2026-11-07 00:00:00

Ticker: RELIANCE.NS
Total rows: 4159
Rows with valid Close: 4159
Rows with missing Close: 0
First valid Close: 2010-01-04 00:00:00
Last valid Close: 2026-11-06 00:00:00

First missing Close observations:


Price,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,



Ticker: JIOFIN.NS
Total rows: 4159
Rows with valid Close: 794
Rows with missing Close: 3365
First valid Close: 2023-08-21 00:00:00
Last valid Close: 2026-11-06 00:00:00

First missing Close observations:


Price,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2010-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Ticker: MAXHEALTH.NS
Total rows: 4159
Rows with valid Close: 1538
Rows with missing Close: 2621
First valid Close: 2020-08-21 00:00:00
Last valid Close: 2026-11-06 00:00:00

First missing Close observations:


Price,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2010-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



NIFTY reference data for the first reported missing dates:


Price,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2010-01-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-02-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-03-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-09-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-11-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-11-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
sample_files = {
    "NIFTY": MACRO_RAW_DIR / "NSEI.parquet",
    "RELIANCE": STOCK_RAW_DIR / "RELIANCE_NS.parquet",
    "JIOFIN": STOCK_RAW_DIR / "JIOFIN_NS.parquet",
    "MAXHEALTH": STOCK_RAW_DIR / "MAXHEALTH_NS.parquet"
}

for name, path in sample_files.items():
    data = pd.read_parquet(path)

    print(f"\n{name}")
    print("Rows:", len(data))
    print("Index type:", type(data.index).__name__)
    print("Invalid dates:", pd.isna(data.index).sum())
    print("First date:", data.index.min())
    print("Last date:", data.index.max())

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(-1)

    if "Close" in data.columns:
        print("Missing Close:", data["Close"].isna().sum())
        print("Valid Close:", data["Close"].notna().sum())


NIFTY
Rows: 4396
Index type: DatetimeIndex
Invalid dates: 0
First date: 2010-01-01 00:00:00
Last date: 2026-11-07 00:00:00
Missing Close: 258
Valid Close: 4138

RELIANCE
Rows: 4159
Index type: DatetimeIndex
Invalid dates: 0
First date: 2010-01-04 00:00:00
Last date: 2026-11-06 00:00:00
Missing Close: 0
Valid Close: 4159

JIOFIN
Rows: 4159
Index type: DatetimeIndex
Invalid dates: 0
First date: 2010-01-04 00:00:00
Last date: 2026-11-06 00:00:00
Missing Close: 3365
Valid Close: 794

MAXHEALTH
Rows: 4159
Index type: DatetimeIndex
Invalid dates: 0
First date: 2010-01-04 00:00:00
Last date: 2026-11-06 00:00:00
Missing Close: 2621
Valid Close: 1538


#### Since we have no Invalid Dates, the missing trading dates are not an issue. 

#### Interpretation
1. Reliance has complete Close data over its downloaded period
2. JIOFIN and MAXHEALTH have missing values since they were listed after 2010. Hence, these are expected pre-listing gaps

In [21]:
# Clean up datasets 

# Output directories
PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CLEAN_STOCK_DIR = PROCESSED_DIR / "stocks"
CLEAN_MACRO_DIR = PROCESSED_DIR / "market_macro"

CLEAN_STOCK_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_MACRO_DIR.mkdir(parents=True, exist_ok=True)

cleaning_log = []

# Exclude unreliable OHLC fields for these macro datasets
CLOSE_ONLY_FILES = {"INR_X", "BZ_F"}

def clean_dataset(file_path, output_dir, dataset_type):

    data = pd.read_parquet(file_path).copy()

    # Flatten Yahoo Finance MultiIndex columns if present
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(-1)

    # Standardize column names
    data.columns = [str(col).strip() for col in data.columns]

    original_rows = len(data)

    # Standardize dates and remove duplicate dates
    data.index = pd.to_datetime(data.index).normalize()
    duplicate_dates = int(data.index.duplicated(keep="last").sum())
    data = data.loc[~data.index.duplicated(keep="last")].copy()

    # Ensure Close is numeric
    data["Close"] = pd.to_numeric(data["Close"], errors="coerce")

    missing_close = int(data["Close"].isna().sum())
    nonpositive_close = int((data["Close"] <= 0).sum())

    # Keep only observations with valid Close prices
    data = data.loc[
        data["Close"].notna() & (data["Close"] > 0)
    ].copy()

    # Exclude unreliable OHLC fields from selected macro datasets
    excluded_columns = []

    if file_path.stem in CLOSE_ONLY_FILES:
        excluded_columns = [
            col for col in ["Open", "High", "Low"]
            if col in data.columns
        ]
        data = data.drop(columns=excluded_columns)

    # Save cleaned data without modifying the raw file
    output_path = output_dir / file_path.name
    data.to_parquet(output_path)

    cleaning_log.append({
        "dataset": file_path.stem,
        "type": dataset_type,
        "original_rows": original_rows,
        "duplicate_dates_removed": duplicate_dates,
        "missing_close_removed": missing_close,
        "nonpositive_close_removed": nonpositive_close,
        "excluded_columns": ", ".join(excluded_columns),
        "clean_rows": len(data),
        "first_date": data.index.min().date(),
        "last_date": data.index.max().date(),
        "output_file": str(output_path)
    })


# Clean stock datasets
for file_path in STOCK_RAW_DIR.glob("*.parquet"):
    clean_dataset(file_path, CLEAN_STOCK_DIR, "stock")


# Clean market and macro datasets
for file_path in MACRO_RAW_DIR.glob("*.parquet"):
    clean_dataset(file_path, CLEAN_MACRO_DIR, "market_macro")


# Save cleaning decisions and results
cleaning_log = pd.DataFrame(cleaning_log)

cleaning_log.to_csv(
    PROCESSED_DIR / "cleaning_log.csv",
    index=False
)

display(cleaning_log)

print("\nTotal datasets cleaned:", len(cleaning_log))
print("Cleaned data directory:", PROCESSED_DIR)


,dataset,type,original_rows,duplicate_dates_removed,missing_close_removed,nonpositive_close_removed,excluded_columns,clean_rows,first_date,last_date,output_file
0,AXISBANK_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
1,JIOFIN_NS,stock,4159,0,3365,0,,794,2023-08-21,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
2,SBILIFE_NS,stock,4159,0,1911,0,,2248,2017-10-03,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
3,HCLTECH_NS,stock,4160,0,0,0,,4160,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
4,SUNPHARMA_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
5,POWERGRID_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
6,TCS_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
7,HINDUNILVR_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
8,ICICIBANK_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...
9,KOTAKBANK_NS,stock,4159,0,0,0,,4159,2010-01-04,2026-11-06,/Users/srisuphachawla/Documents/DataScience_Pr...



Total datasets cleaned: 55
Cleaned data directory: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/notebooks/data/processed


In [22]:
print("Cleaning log columns:")
print(cleaning_log.columns.tolist())

print("\nTotal datasets:", len(cleaning_log))
print("\nDatasets by type:")
print(cleaning_log["type"].value_counts())

print("\nCleaning summary:")
display(
    cleaning_log[
        [
            "dataset",
            "type",
            "original_rows",
            "duplicate_dates_removed",
            "missing_close_removed",
            "nonpositive_close_removed",
            "clean_rows"
        ]
    ].to_string(index=False)
)

Cleaning log columns:
['dataset', 'type', 'original_rows', 'duplicate_dates_removed', 'missing_close_removed', 'nonpositive_close_removed', 'excluded_columns', 'clean_rows', 'first_date', 'last_date', 'output_file']

Total datasets: 55

Datasets by type:
type
stock           50
market_macro     5
Name: count, dtype: int64

Cleaning summary:


'      dataset         type  original_rows  duplicate_dates_removed  missing_close_removed  nonpositive_close_removed  clean_rows\n  AXISBANK_NS        stock           4159                        0                      0                          0        4159\n    JIOFIN_NS        stock           4159                        0                   3365                          0         794\n   SBILIFE_NS        stock           4159                        0                   1911                          0        2248\n   HCLTECH_NS        stock           4160                        0                      0                          0        4160\n SUNPHARMA_NS        stock           4159                        0                      0                          0        4159\n POWERGRID_NS        stock           4159                        0                      0                          0        4159\n       TCS_NS        stock           4159                        0                      0

In [24]:
# CHECK MISSING OBSERVATIONS AND LARGE PRICE MOVEMENTS
# Reference trading calendar
nifty = pd.read_parquet(CLEAN_MACRO_DIR / "NSEI.parquet")

if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.get_level_values(-1)

nifty.index = pd.to_datetime(nifty.index).normalize()

nifty_dates = nifty.index[
    nifty["Close"].notna() & (nifty["Close"] > 0)
].unique().sort_values()

missing_results = []
jump_results = []

for file_path in sorted(CLEAN_STOCK_DIR.glob("*.parquet")):

    data = pd.read_parquet(file_path)

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(-1)

    data.index = pd.to_datetime(data.index).normalize()
    close = data["Close"].dropna().astype(float).sort_index()

    if close.empty:
        continue

    # Identify missing observations within the stock's history
    expected_dates = nifty_dates[
        (nifty_dates >= close.index.min()) &
        (nifty_dates <= close.index.max())
    ]

    missing_dates = expected_dates.difference(close.index)

    missing_results.append({
        "ticker": file_path.stem,
        "missing_sessions": len(missing_dates),
        "missing_date_examples": [
            str(d.date()) for d in missing_dates[:10]
        ]
    })

    # Calculate returns only between consecutive NIFTY sessions
    aligned_close = close.reindex(expected_dates)

    previous_close = aligned_close.shift(1)

    daily_returns = aligned_close.div(previous_close).sub(1)

    consecutive_observations = (
        aligned_close.notna() & previous_close.notna()
    )

    daily_returns = daily_returns[
        consecutive_observations
    ]

    # Flag movements exceeding +/-15%
    large_moves = daily_returns[daily_returns.abs() > 0.15]

    for date, daily_return in large_moves.items():
        jump_results.append({
            "ticker": file_path.stem,
            "date": date.date(),
            "daily_return_pct": round(daily_return * 100, 2),
            "previous_close": previous_close.loc[date],
            "current_close": aligned_close.loc[date]
        })


# Save audit results
missing_audit = pd.DataFrame(missing_results)

large_return_audit = pd.DataFrame(
    jump_results,
    columns=[
        "ticker",
        "date",
        "daily_return_pct",
        "previous_close",
        "current_close"
    ]
)

missing_audit.to_csv(
    PROCESSED_DIR / "missing_observation_audit.csv",
    index=False
)

large_return_audit = large_return_audit.sort_values(
    "daily_return_pct",
    key=lambda x: x.abs(),
    ascending=False
)

large_return_audit.to_csv(
    PROCESSED_DIR / "large_price_movements.csv",
    index=False
)

# Display only actionable results
print("Stocks with missing reference-calendar sessions:",
      (missing_audit["missing_sessions"] > 0).sum())

print("Total flagged missing sessions:",
      missing_audit["missing_sessions"].sum())

print("\nLargest price movements exceeding +/-15%:")
display(large_return_audit.head(30))


Stocks with missing reference-calendar sessions: 0
Total flagged missing sessions: 0

Largest price movements exceeding +/-15%:


,ticker,date,daily_return_pct,previous_close,current_close
63,NESTLEIND_NS,2010-01-08,-76.33,527.049988,124.732498
85,TMPV_NS,2025-10-14,-40.15,660.750000,395.450012
3,ADANIENT_NS,2015-06-03,-38.75,97.636215,59.802910
81,TMPV_NS,2019-10-29,35.95,126.849998,172.449997
89,TRENT_NS,2026-01-01,-33.05,4279.000000,2864.933350
12,ADANIENT_NS,2023-02-01,-28.20,2973.899902,2135.350098
27,AXISBANK_NS,2020-03-23,-27.91,428.149994,308.649994
69,SBIN_NS,2017-10-25,27.69,254.449997,324.899994
8,ADANIENT_NS,2019-05-20,27.37,119.300003,151.949997
13,ADANIENT_NS,2023-02-02,-26.70,2135.350098,1565.250000


In [26]:
# INSPECT LARGE PRICE MOVEMENTS

if large_return_audit.empty:
    print("No consecutive-session returns exceeded +/-15%.")
else:
    print("Total flagged movements:", len(large_return_audit))

    print("\nMovements by stock:")
    display(
        large_return_audit.groupby("ticker")
        .agg(
            flagged_movements=("date", "count"),
            largest_return_pct=("daily_return_pct",
                                lambda x: x.loc[x.abs().idxmax()])
        )
        .sort_values("flagged_movements", ascending=False)
    )

    print("\nFlagged observations:")
    display(large_return_audit.head(30))

    

Total flagged movements: 90

Movements by stock:


,flagged_movements,largest_return_pct
ticker,,
ADANIENT_NS,20,-38.75
TMPV_NS,6,-40.15
SHRIRAMFIN_NS,6,-22.18
BSE_NS,5,18.60
ADANIPORTS_NS,5,-21.15
ONGC_NS,4,18.51
TRENT_NS,4,-33.05
BAJAJFINSV_NS,4,-25.86
BEL_NS,3,-19.80



Flagged observations:


,ticker,date,daily_return_pct,previous_close,current_close
63,NESTLEIND_NS,2010-01-08,-76.33,527.049988,124.732498
85,TMPV_NS,2025-10-14,-40.15,660.750000,395.450012
3,ADANIENT_NS,2015-06-03,-38.75,97.636215,59.802910
81,TMPV_NS,2019-10-29,35.95,126.849998,172.449997
89,TRENT_NS,2026-01-01,-33.05,4279.000000,2864.933350
12,ADANIENT_NS,2023-02-01,-28.20,2973.899902,2135.350098
27,AXISBANK_NS,2020-03-23,-27.91,428.149994,308.649994
69,SBIN_NS,2017-10-25,27.69,254.449997,324.899994
8,ADANIENT_NS,2019-05-20,27.37,119.300003,151.949997
13,ADANIENT_NS,2023-02-02,-26.70,2135.350098,1565.250000


In [28]:
# Checking Macro Data Alignment with Nifty

# NIFTY reference trading calendar
nifty = pd.read_parquet(CLEAN_MACRO_DIR / "NSEI.parquet")

nifty_dates = pd.DatetimeIndex(nifty.index)

if nifty_dates.tz is not None:
    nifty_dates = nifty_dates.tz_localize(None)

nifty_dates = nifty_dates.normalize()

nifty_dates = nifty_dates[
    nifty["Close"].notna().to_numpy()
]

alignment_results = []

for file_path in sorted(CLEAN_MACRO_DIR.glob("*.parquet")):

    if file_path.stem == "NSEI":
        continue

    data = pd.read_parquet(file_path)

    valid_dates = pd.DatetimeIndex(
        data.index[data["Close"].notna()]
    )

    if valid_dates.tz is not None:
        valid_dates = valid_dates.tz_localize(None)

    valid_dates = valid_dates.normalize()

    matched_dates = nifty_dates.intersection(valid_dates)

    missing_dates = nifty_dates.difference(valid_dates)

    alignment_results.append({
        "dataset": file_path.stem,
        "matched_nifty_sessions": len(matched_dates),
        "missing_nifty_sessions": len(missing_dates),
        "first_missing_dates": [
            str(d.date()) for d in missing_dates[:5]
        ]
    })

macro_alignment_audit = pd.DataFrame(alignment_results)

display(macro_alignment_audit)

,dataset,matched_nifty_sessions,missing_nifty_sessions,first_missing_dates
0,BZ_F,3991,147,"[2010-01-18, 2010-02-15, 2010-05-04, 2010-05-3..."
1,INDIAVIX,4121,17,"[2020-01-01, 2021-01-01, 2021-01-22, 2021-02-1..."
2,INR_X,4129,9,"[2011-04-15, 2013-10-08, 2017-07-11, 2017-11-1..."
3,TNX,4020,118,"[2010-01-18, 2010-02-15, 2010-05-31, 2010-07-0..."


In [29]:
# Align Macroeconomic data to Nifty Trading Calendar

# Dataset filename and conservative lag in NIFTY sessions
macro_config = {
    "INDIA_VIX": {"file": "INDIAVIX.parquet", "lag": 0},
    "USD_INR": {"file": "INR_X.parquet", "lag": 1},
    "BRENT_CRUDE": {"file": "BZ_F.parquet", "lag": 1},
    "US_10Y_YIELD": {"file": "TNX.parquet", "lag": 1}
}

# Load NIFTY reference calendar
nifty = pd.read_parquet(CLEAN_MACRO_DIR / "NSEI.parquet")

if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.get_level_values(-1)

nifty.index = pd.to_datetime(nifty.index)

if nifty.index.tz is not None:
    nifty.index = nifty.index.tz_localize(None)

nifty.index = nifty.index.normalize()

nifty = nifty[
    nifty["Close"].notna() & (nifty["Close"] > 0)
].sort_index()

nifty_dates = pd.DatetimeIndex(nifty.index.unique())

# Initialize aligned dataset
aligned_macro = pd.DataFrame(index=nifty_dates)

aligned_macro["NIFTY_50_Close"] = (
    nifty["Close"].groupby(nifty.index).last().reindex(nifty_dates)
)

# Align each macroeconomic variable
for name, config in macro_config.items():

    data = pd.read_parquet(
        CLEAN_MACRO_DIR / config["file"]
    )

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(-1)

    data.index = pd.to_datetime(data.index)

    if data.index.tz is not None:
        data.index = data.index.tz_localize(None)

    data.index = data.index.normalize()

    source = (
        data[["Close"]]
        .dropna()
        .sort_index()
    )

    source = source[
        source["Close"] > 0
    ]

    source = source[
        ~source.index.duplicated(keep="last")
    ]

    source_dates = pd.DatetimeIndex(source.index)

    # Determine the information cutoff for each NIFTY session
    cutoff_dates = nifty_dates.to_series().shift(
        config["lag"]
    )

    cutoff_dates = pd.DatetimeIndex(cutoff_dates)

    # Find the latest source observation on or before each cutoff
    valid_cutoffs = ~cutoff_dates.isna()

    positions = source_dates.searchsorted(
        cutoff_dates[valid_cutoffs],
        side="right"
    ) - 1

    values = pd.Series(
        float("nan"),
        index=nifty_dates,
        dtype="float64"
    )

    source_date_used = pd.Series(
        pd.NaT,
        index=nifty_dates,
        dtype="datetime64[ns]"
    )

    available = positions >= 0

    target_dates = nifty_dates[valid_cutoffs][available]
    source_positions = positions[available]

    values.loc[target_dates] = (
        source["Close"].iloc[source_positions].to_numpy()
    )

    source_date_used.loc[target_dates] = (
        source_dates[source_positions]
    )

    aligned_macro[f"{name}_Close"] = values
    aligned_macro[f"{name}_Source_Date"] = source_date_used

# Save aligned dataset
output_path = PROCESSED_DIR / "market_macro_aligned.parquet"

aligned_macro.to_parquet(output_path)

print("Aligned macroeconomic dataset created.")
print("Shape:", aligned_macro.shape)
print("Output:", output_path)

display(aligned_macro.head())

Aligned macroeconomic dataset created.
Shape: (4138, 9)
Output: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/notebooks/data/processed/market_macro_aligned.parquet


,NIFTY_50_Close,INDIA_VIX_Close,INDIA_VIX_Source_Date,USD_INR_Close,USD_INR_Source_Date,BRENT_CRUDE_Close,BRENT_CRUDE_Source_Date,US_10Y_YIELD_Close,US_10Y_YIELD_Source_Date
Date,,,,,,,,,
2010-01-04,5232.200195,23.639999,2010-01-04,NaN,NaT,NaN,NaT,NaN,NaT
2010-01-05,5277.899902,22.270000,2010-01-05,46.287998,2010-01-04,80.120003,2010-01-04,3.841,2010-01-04
2010-01-06,5281.799805,22.120001,2010-01-06,46.119999,2010-01-05,80.589996,2010-01-05,3.755,2010-01-05
2010-01-07,5263.100098,22.500000,2010-01-07,45.720001,2010-01-06,81.889999,2010-01-06,3.808,2010-01-06
2010-01-08,5244.750000,22.570000,2010-01-08,45.688000,2010-01-07,81.510002,2010-01-07,3.822,2010-01-07


In [30]:
# Validate Macroeconomic alignment

alignment_checks = []

for name, config in macro_config.items():

    source_dates = aligned_macro[
        f"{name}_Source_Date"
    ]

    cutoff_dates = pd.Series(
        nifty_dates,
        index=nifty_dates
    ).shift(config["lag"])

    invalid = (
        source_dates.notna()
        & (source_dates > cutoff_dates)
    )

    alignment_checks.append({
        "dataset": name,
        "lag_sessions": config["lag"],
        "available_values": source_dates.notna().sum(),
        "missing_values": source_dates.isna().sum(),
        "future_information_violations": invalid.sum()
    })

alignment_validation = pd.DataFrame(alignment_checks)

display(alignment_validation)

assert (
    alignment_validation["future_information_violations"].sum() == 0
), "Future information detected in macroeconomic alignment."

,dataset,lag_sessions,available_values,missing_values,future_information_violations
0,INDIA_VIX,0,4138,0,0
1,USD_INR,1,4137,1,0
2,BRENT_CRUDE,1,4137,1,0
3,US_10Y_YIELD,1,4137,1,0
